# RASA mit Audio

Ziele der heutigen Übung:
- Anbindung eines Spracherkenners (ASR) an einen textbasierten Dialogmanager (RASA)
- Anbindung einer Sprachausgabe (TTS) an einen Dialogmanager (RASA)

### Happy Path Pizza Dialog der letzten Übung

nehme dir etwas Zeit, um mindestens einen Dialogablauf der Pizza Bestellung aus letzter Übung zu implementieren. Nachdem ein Dialogablauf funktioniert, mache in diesem Notebok weiter, um einen Spracherkenner und eine Sprachausgabe an die RASA-API anzubinden.

### ASR

In Session 3 hast du mit der SpechRecognition Bibliothek die Transkription eines Audiofiles einerseits mit der Google Webservice API und andererseits mit Vosk implementiert. Den Google Webservice Code möchten wir hier nutzen, um eine Sprachäußerung aufzunehmen, zu transkribieren und anschließend an RASA zu schicken. Zuerst implementieren wir die Audioaufnahme und Transkription in einer Funktion:

In [ ]:
import speech_recognition as sr

recognizer = sr.Recognizer()

def listen() -> str:
    text = ""
    with sr.Microphone() as source:
        print("Jetzt sprechen...")
        audio_data = recognizer.listen(source)
        print("Daten aufgenommen")

    try:
        text = recognizer.recognize_google(audio_data, language='de-DE')
        #text = "Ich möchte eine vegetarische Pizza bestellen"
        print("You said:", text)
    except sr.RequestError:
        print("Error: Could not access Google Web Speech API;")
    except sr.UnknownValueError:
        print("Sorry, I do not understand")
    return text



Anschließend implementieren wir eine Funktion, mit welcher wir Text an den RASA-Server schicken können. RASA stellt eine REST-API zur Verfügung, welche wird dafür nutzen können. Anstatt mit ```rasa shell``` starten wir den Server in einem Terminal mit:

```rasa run --enable-api connector rest```

In [ ]:
import requests

def send_to_rasa(text: str):
    api_url = "http://localhost:5005/webhooks/rest/webhook"
    # with sender we define one user, thus we have remaining 
    post_data = {"sender": "user_123", "message": text}
    response = requests.post(
            api_url,
            json=post_data,
            timeout=10,
        )
    return response.json()

### TTS

Abschließend benötigen wir noch eine Sprachausgabe. Wie in der Vorlesung gelernt, gibt es verschiedene Varianten: von voraufgenommenen Audiofiles, über phonem-basierte Modelle bis hin zu großen DeepLearning-basierten Modellen.

Um den Download und die Rechenleistung nicht zu sehr zu beanspruchen, nehmen wir PiperTTS.


In [ ]:
pip install piper-tts sounddevice numpy

Mögliche Stimmen mit Samples findest du auf folgender Seite:
https://piper.ttstool.com/

Lade dir eine Stimme deiner Wahl herunter und speichere sie in den Unterordner ```models``` (es müssen 2 Dateien sein, ein ONNX Model und eine JSON Konfigurationsdatei)

Anschließend können wir uns eine Funktion schreiben, die einen Text synthetisiert und als Audio ausgibt:

In [55]:
import numpy as np
import sounddevice as sd
from piper.voice import PiperVoice

# 1. Pfade zu den Modell-Dateien definieren
model_path = "models/de_DE-thorsten_emotional-medium.onnx"
config_path = "models/de_DE-thorsten_emotional-medium.onnx.json"

# 2. Piper-Stimme laden
print("Lade Sprachmodell...")
voice = PiperVoice.load(model_path, config_path=config_path)

def speak(text: str):

    stream = sd.OutputStream(
        samplerate=voice.config.sample_rate, 
        channels=1, 
        dtype="int16"
    )

    # Stream starten und Audio-Chunks direkt abspielen
    with stream:
        # voice.synthesize_stream liefert rohe PCM-Bytes in Echtzeit
        for chunk in voice.synthesize(text):
            # Die rohen Bytes stecken im .audio_int16_bytes Attribut des Chunks
            audio_bytes = chunk.audio_int16_bytes
            
            # Bytes in ein NumPy-Array (int16) umwandeln
            audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
            
            # Den aktuellen Audio-Chunk abspielen
            stream.write(audio_data)

Lade Sprachmodell...


In [56]:
speak("Hallo, ich kann sagen was du willst.")

### Dialog Loop

Nun müssen wir die geschriebenen Funktionen noch zu einem Programmablauf zusammen fügen und haben bereits ein einfaches Dialogsystem mit Sprachein- und -ausgabe :).

In [ ]:
while True:
    text = listen()
    if len(text) > 0:
        response = send_to_rasa(text)
        if len(response) > 0 and 'text' in response[0]:
            speak(response[0]['text'])
        else:
            speak("Ich verstehe nicht, bitte wiederholen.")

Jetz sprechen...
Daten aufgenommen
You said: ich möchte eine vegetarische Pizza bestellen
Jetz sprechen...
Daten aufgenommen
You said: ja
Jetz sprechen...
Daten aufgenommen
You said: ja
